# Udaplay Project
## Part 01 - Offline RAG

In this part we build a **vector database** (ChromaDB) from 15 game JSON files in `games/`.

Plain-English plan:
1. Load the API keys from `.env`.
2. Create a database on disk (`chromadb/`) with a collection called `udaplay`.
3. Turn each game file into one text document and add it to the collection.
4. Test it: ask a question and check the right game comes back.

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
from dotenv import load_dotenv, find_dotenv

# find_dotenv() finds the .env file in this folder (or a parent folder);
# override=True makes the values in .env win over anything already set.
load_dotenv(find_dotenv(), override=True)

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"
# Web search uses OpenAI's built-in web search tool with the same key,
# so a TAVILY_API_KEY is not needed.
print("Environment variables are set.")

Environment variables are set.


In [3]:
import json
import chromadb
from chromadb.utils import embedding_functions

### VectorDB Instance
`PersistentClient` saves the database to the `chromadb/` folder, so Part 2 can load it.

In [4]:
# Keep this path: Part 2 loads the database from "chromadb" (inside project/starter)
chroma_client = chromadb.PersistentClient(path="chromadb")

### Collection
A collection is like a table. We tell it which embedding function to use.

In [5]:
# The embedding function turns text into numbers (vectors) so we can compare meaning.
# Part 2 rebuilds this exact same function, so keep it identical in both notebooks.
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key_env_var="OPENAI_API_KEY",
    api_base=os.getenv("OPENAI_BASE_URL"),
    model_name="text-embedding-ada-002",
)

In [6]:
# Start clean so re-running this notebook doesn't fail on duplicate IDs
if "udaplay" in [c.name for c in chroma_client.list_collections()]:
    chroma_client.delete_collection("udaplay")

# Keep this name: Part 2 loads the collection "udaplay"
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn,
)
print("Collection ready:", collection.name)

Collection ready: udaplay


### Add documents
Each game becomes one document. The **text** we embed includes name, platform, year, genre, publisher and description, so questions about any of these can match. The full JSON is also stored as **metadata**.

In [7]:
# Run this notebook from project/starter: the game files are in ./games
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # The text that gets embedded (what semantic search compares against)
    content = (
        f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - "
        f"Genre: {game['Genre']}. Publisher: {game['Publisher']}. {game['Description']}"
    )

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game],
    )

print("Documents in collection:", collection.count())

Documents in collection: 15


### Test semantic search
We ask questions in plain English. The closest games come back first (smaller **distance** = more similar).

In [8]:
def search(question, n_results=3):
    results = collection.query(query_texts=[question], n_results=n_results)
    print(f"Q: {question}")
    for meta, dist in zip(results["metadatas"][0], results["distances"][0]):
        print(f"   {dist:.3f}  {meta['Name']} | {meta['Platform']} | {meta['YearOfRelease']} | {meta['Publisher']}")
    print()

search("Who published Grand Theft Auto: San Andreas?")
search("When was Pokémon Gold and Silver released?")
search("What platform was Super Mario 64 launched on?")
search("a racing game on PlayStation")

Q: Who published Grand Theft Auto: San Andreas?
   0.123  Grand Theft Auto: San Andreas | PlayStation 2 | 2004 | Rockstar Games
   0.210  Gran Turismo | PlayStation 1 | 1997 | Sony Computer Entertainment
   0.213  Gran Turismo 5 | PlayStation 3 | 2010 | Sony Computer Entertainment



Q: When was Pokémon Gold and Silver released?
   0.118  Pokémon Gold and Silver | Game Boy Color | 1999 | Nintendo
   0.151  Pokémon Ruby and Sapphire | Game Boy Advance | 2002 | Nintendo
   0.229  Super Mario 64 | Nintendo 64 | 1996 | Nintendo



Q: What platform was Super Mario 64 launched on?
   0.108  Super Mario 64 | Nintendo 64 | 1996 | Nintendo
   0.151  Super Mario World | Super Nintendo Entertainment System (SNES) | 1990 | Nintendo
   0.205  Mario Kart 8 Deluxe | Nintendo Switch | 2017 | Nintendo



Q: a racing game on PlayStation
   0.118  Gran Turismo 5 | PlayStation 3 | 2010 | Sony Computer Entertainment
   0.124  Gran Turismo | PlayStation 1 | 1997 | Sony Computer Entertainment
   0.173  Mario Kart 8 Deluxe | Nintendo Switch | 2017 | Nintendo



The right game is at the top for each question, so the database works. Part 2 uses it as the agent's first source of knowledge.